# 02. Feature Engineering Pipeline

본 노트북은 초기 100 사이클 데이터를 기반으로 배터리 수명을 예측하기 위한 핵심 물리화학적 피처들을 추출하고 정제합니다.
- $\Delta Q_{100-10}(V)$ 전압 변환 피처
- 방전 용량 감쇠율 및 선형 피팅 피처
- 충전 조건 및 발열/내부저항 피처
- Level 1 / Level 2 / Level 3 피처 셋 구성


In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 상위 폴더 src 모듈 참조
sys.path.append('../src')
from preprocess import load_processed_data
from features import extract_features, FEATURE_SETS

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = ['AppleGothic', 'NanumGothic', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False



## 1. 데이터 로드 및 피처 자동 추출


In [ ]:
meta, summary, qdlin = load_processed_data('../processed_data')
df_features = extract_features(meta, summary, qdlin)
print(f'✅ 추출 완료된 데이터 크기: {df_features.shape}')
df_features.head()


## 2. 피처 레벨 정의 및 기술 통계


In [ ]:
for level, cols in FEATURE_SETS.items():
    print(f'=== {level} ({len(cols)}개 피처) ===')
    print(cols)
    print()
df_features[FEATURE_SETS["Level 3 (Full)"]].describe().T


## 3. 피처 간 상관계수 히트맵


In [ ]:
corr_cols = ['cycle_life', 'log_cycle_life'] + FEATURE_SETS['Level 3 (Full)']
corr_matrix = df_features[corr_cols].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Feature Correlation Matrix with Battery Cycle Life', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('🏆 수명(log_cycle_life)과의 상관계수 상위 피처:')
display(corr_matrix['log_cycle_life'].sort_values(key=abs, ascending=False).to_frame())
